In [0]:
%sql
CREATE or REPLACE TABLE `medallion_architecture`.`gold`.`gold_facility_contact` (
  -- Identity
  `contact_id` STRING COMMENT 'Stable internal contact ID (generated)',
  `facility_id` STRING COMMENT 'Foreign key to gold_facility.facility_id',
  
  -- Contact Details
  `contact_type` STRING COMMENT 'Phone, email, website, official_website, facebook, etc.',
  `contact_value` STRING COMMENT 'Cleaned contact value',
  
  -- Classification
  `is_official` BOOLEAN COMMENT 'Whether the contact came from an official source',
  `is_primary` BOOLEAN COMMENT 'Best contact to show first',
  `department` STRING COMMENT 'Department if contact is specific (emergency, OPD, billing, etc.)',
  
  -- Quality Metrics
  `source_rank` INT COMMENT 'Trust rank of the source (1=highest)',
  `confidence_score` DOUBLE COMMENT 'Confidence in the contact value (0-1)',
  `last_seen_at` TIMESTAMP COMMENT 'Most recent date this contact was observed',
  `contact_quality_flags` ARRAY<STRING> COMMENT 'Duplicate, invalid format, likely aggregator, outdated, etc.',
  
  -- Audit
  `created_at` TIMESTAMP COMMENT 'Record creation timestamp',
  `updated_at` TIMESTAMP COMMENT 'Record last update timestamp'
)
COMMENT 'Gold layer contact table for facility contact methods. One row per facility per contact method.';


In [0]:
%sql
INSERT INTO `medallion_architecture`.`gold`.`gold_facility_contact`

-- Official Phone contacts
SELECT
  CONCAT(`unique_id`, '_phone_official') AS `contact_id`,
  `unique_id` AS `facility_id`,
  'phone' AS `contact_type`,
  TRIM(`official_phone`) AS `contact_value`,
  TRUE AS `is_official`,
  TRUE AS `is_primary`,
  CAST(NULL AS STRING) AS `department`,
  1 AS `source_rank`,
  CASE 
    WHEN `official_phone` RLIKE '^\\+91[0-9]{10}$' THEN 1.0
    WHEN `official_phone` RLIKE '^\\+[0-9]{10,15}$' THEN 0.9
    WHEN `official_phone` LIKE '+91%' THEN 0.8
    ELSE 0.6
  END AS `confidence_score`,
  COALESCE(`most_recent_social_media_post_date`, `recency_of_page_update`, `silver_ingested_at`) AS `last_seen_at`,
  ARRAY_COMPACT(ARRAY(
    CASE WHEN `official_phone` NOT LIKE '+%' THEN 'missing_country_code' ELSE NULL END,
    CASE WHEN `official_phone` RLIKE '^[0-9-]+$' THEN 'invalid_format' ELSE NULL END
  )) AS `contact_quality_flags`,
  CURRENT_TIMESTAMP() AS `created_at`,
  CURRENT_TIMESTAMP() AS `updated_at`
FROM `medallion_architecture`.`silver`.`facilities`
WHERE `official_phone` IS NOT NULL
  AND `official_phone` != 'null'
  AND `official_phone` NOT LIKE '%true%'
  AND LENGTH(TRIM(`official_phone`)) > 5

UNION ALL

-- Phone numbers from array (exploded)
SELECT
  CONCAT(`unique_id`, '_phone_', MD5(phone_value)) AS `contact_id`,
  `unique_id` AS `facility_id`,
  'phone' AS `contact_type`,
  TRIM(phone_value) AS `contact_value`,
  FALSE AS `is_official`,
  FALSE AS `is_primary`,
  CAST(NULL AS STRING) AS `department`,
  2 AS `source_rank`,
  CASE 
    WHEN phone_value RLIKE '^\\+91[0-9]{10}$' THEN 0.9
    WHEN phone_value RLIKE '^\\+[0-9]{10,15}$' THEN 0.8
    WHEN phone_value LIKE '+91%' THEN 0.7
    ELSE 0.5
  END AS `confidence_score`,
  COALESCE(`most_recent_social_media_post_date`, `recency_of_page_update`, `silver_ingested_at`) AS `last_seen_at`,
  ARRAY_COMPACT(ARRAY(
    CASE WHEN phone_value NOT LIKE '+%' THEN 'missing_country_code' ELSE NULL END,
    CASE WHEN phone_value RLIKE '^[0-9-]+$' THEN 'invalid_format' ELSE NULL END,
    CASE WHEN phone_value = `official_phone` THEN 'duplicate_of_official' ELSE NULL END
  )) AS `contact_quality_flags`,
  CURRENT_TIMESTAMP() AS `created_at`,
  CURRENT_TIMESTAMP() AS `updated_at`
FROM `medallion_architecture`.`silver`.`facilities`
LATERAL VIEW EXPLODE(FROM_JSON(`phone_numbers`, 'array<string>')) AS phone_value
WHERE `phone_numbers` IS NOT NULL
  AND phone_value IS NOT NULL
  AND LENGTH(TRIM(phone_value)) > 5
  AND phone_value RLIKE '^\\+[0-9]'

UNION ALL

-- Email contacts
SELECT
  CONCAT(`unique_id`, '_email_', MD5(`email`)) AS `contact_id`,
  `unique_id` AS `facility_id`,
  'email' AS `contact_type`,
  LOWER(TRIM(`email`)) AS `contact_value`,
  TRUE AS `is_official`,
  TRUE AS `is_primary`,
  CAST(NULL AS STRING) AS `department`,
  1 AS `source_rank`,
  CASE 
    WHEN `email` RLIKE '^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\\.[a-zA-Z]{2,}$' THEN 1.0
    WHEN `email` LIKE '%@%' THEN 0.7
    ELSE 0.3
  END AS `confidence_score`,
  COALESCE(`most_recent_social_media_post_date`, `recency_of_page_update`, `silver_ingested_at`) AS `last_seen_at`,
  ARRAY_COMPACT(ARRAY(
    CASE WHEN `email` NOT LIKE '%@%' THEN 'invalid_email_format' ELSE NULL END,
    CASE WHEN `email` LIKE '%[%' OR `email` LIKE '%]%' THEN 'contains_array_chars' ELSE NULL END
  )) AS `contact_quality_flags`,
  CURRENT_TIMESTAMP() AS `created_at`,
  CURRENT_TIMESTAMP() AS `updated_at`
FROM `medallion_architecture`.`silver`.`facilities`
WHERE `email` IS NOT NULL
  AND `email` != 'null'
  AND LENGTH(TRIM(`email`)) > 5
  AND `email` LIKE '%@%'
  AND `email` NOT LIKE '%[%'

UNION ALL

-- Official Website contacts
SELECT
  CONCAT(`unique_id`, '_website_official') AS `contact_id`,
  `unique_id` AS `facility_id`,
  'website' AS `contact_type`,
  LOWER(TRIM(`official_website`)) AS `contact_value`,
  TRUE AS `is_official`,
  TRUE AS `is_primary`,
  CAST(NULL AS STRING) AS `department`,
  1 AS `source_rank`,
  CASE 
    WHEN `official_website` RLIKE '^[a-zA-Z0-9].-]+\\.[a-zA-Z]{2,}$' THEN 1.0
    WHEN `official_website` LIKE '%.%' THEN 0.8
    ELSE 0.5
  END AS `confidence_score`,
  COALESCE(`most_recent_social_media_post_date`, `recency_of_page_update`, `silver_ingested_at`) AS `last_seen_at`,
  ARRAY_COMPACT(ARRAY(
    CASE WHEN `official_website` NOT LIKE '%.%' THEN 'invalid_domain_format' ELSE NULL END,
    CASE WHEN `official_website` = 'true' OR `official_website` = 'null' THEN 'invalid_value' ELSE NULL END
  )) AS `contact_quality_flags`,
  CURRENT_TIMESTAMP() AS `created_at`,
  CURRENT_TIMESTAMP() AS `updated_at`
FROM `medallion_architecture`.`silver`.`facilities`
WHERE `official_website` IS NOT NULL
  AND `official_website` != 'null'
  AND `official_website` != 'true'
  AND LENGTH(TRIM(`official_website`)) > 3
  AND `official_website` LIKE '%.%'

UNION ALL

-- Facebook contacts
SELECT
  CONCAT(`unique_id`, '_facebook') AS `contact_id`,
  `unique_id` AS `facility_id`,
  'facebook' AS `contact_type`,
  TRIM(`facebook_link`) AS `contact_value`,
  TRUE AS `is_official`,
  FALSE AS `is_primary`,
  CAST(NULL AS STRING) AS `department`,
  2 AS `source_rank`,
  CASE 
    WHEN `facebook_link` LIKE 'https\://www.facebook.com/%' THEN 1.0
    WHEN `facebook_link` LIKE '%facebook.com%' THEN 0.8
    ELSE 0.5
  END AS `confidence_score`,
  COALESCE(`most_recent_social_media_post_date`, `recency_of_page_update`, `silver_ingested_at`) AS `last_seen_at`,
  ARRAY_COMPACT(ARRAY(
    CASE WHEN `facebook_link` NOT LIKE '%facebook.com%' THEN 'invalid_facebook_url' ELSE NULL END
  )) AS `contact_quality_flags`,
  CURRENT_TIMESTAMP() AS `created_at`,
  CURRENT_TIMESTAMP() AS `updated_at`
FROM `medallion_architecture`.`silver`.`facilities`
WHERE `facebook_link` IS NOT NULL
  AND `facebook_link` != 'null'
  AND LENGTH(TRIM(`facebook_link`)) > 10
  AND `facebook_link` LIKE '%facebook%';


num_affected_rows,num_inserted_rows
131799,131799
